In [5]:
import serial
import time

# ⚠️ 請將 'COM4' 修改為您在裝置管理員看到的實際 COM Port 名稱
COM_PORT = 'COM4'
BAUD_RATE = 115200  # 必須與 STM32CubeMX 中設定的波特率一致[cite: 4]

try:
    # 建立序列埠連線
    ser = serial.Serial(
        port=COM_PORT,
        baudrate=BAUD_RATE,
        bytesize=serial.EIGHTBITS,
        parity=serial.PARITY_NONE,
        stopbits=serial.STOPBITS_ONE,
        timeout=1
    )
    
    print(f"✅ 成功連接至 {COM_PORT}！開始監聽 STM32 數據...")
    time.sleep(1)  # 等待連線穩定

    while True:
        # 只有在有新資料近來時才讀取與處理
        if ser.in_waiting > 0:
            raw_data = ser.readline()
            response = raw_data.decode('utf-8', errors='ignore').strip()
        
            # 必須放在 if ser.in_waiting > 0 內部（注意縮排！）
            if "TIME:" in response:
                # 1. 抓出數字部分
                us_str = response.split("TIME:")[1].split("_US")[0]
                us_val = float(us_str)
                
                # 2. 轉換成秒 (除以 1,000,000)
                sec_val = us_val / 1000000.0
                
                # 3. 印出以秒為單位的結果 (保留 4 位小數)
                print(f"🏎️ 反應成績：{sec_val:.4f} 秒 ({us_val:.0f} μs)")
                
            elif "FALSE_START" in response:
                print("⚠️ 偷跑違規！")
            
            elif "HELLO_STM32" in response:
                print("👋 收到連線確認訊號：HELLO_STM32")

except serial.SerialException as e:
    print(f"❌ 序列埠連接失敗（請檢查 COM Port 是否正確或被其他程式佔用）: {e}")

except KeyboardInterrupt:
    print("\n👋 程式已手動終止。")

finally:
    if 'ser' in locals() and ser.is_open:
        ser.close()
        print("🔒 序列埠已安全關閉。")

✅ 成功連接至 COM4！開始監聽 STM32 數據...
⚠️ 偷跑違規！
⚠️ 偷跑違規！
🏎️ 反應成績：0.2959 秒 (295948 μs)
❌ 序列埠連接失敗（請檢查 COM Port 是否正確或被其他程式佔用）: ClearCommError failed (OSError(22, '因為執行緒結束或應用程式要求，所以已中止 I/O 操作。', None, 995))
🔒 序列埠已安全關閉。
